# Exercise 01 — Workbook: your own protein

This notebook is **deliberately almost empty.** Its companion, **`ex01_guide.ipynb`**,
walks the entire analysis through on one protein (1FSZ, FtsZ) with every cell filled in.
Here you rebuild that analysis on a protein of **your own** choosing.

**Read `ex01_guide.ipynb` first.** Then work here, with it open beside you.

## How to work in this notebook

- **Copy code across from the guide and adapt it.** `extract_structure_info`,
  `find_best_structure`, `classify_quality` and `parse_fasta_alignment` are all written
  to be generic — they work on any protein, not just 1FSZ. Reusing them is the point.
- **Structure the notebook however you like.** The section headers below are a checklist
  of what has to be here, not a cell-by-cell template. Add cells, split them, reorder
  them, delete these prompts once you've answered them.
- **The reasoning is what's graded, not the code.** Every section below asks for a
  written conclusion as well as output. A notebook full of correct plots with no
  interpretation scores badly; one with an honest, well-argued interpretation of a
  messy result scores well.

## Using AI Tools

You may use AI assistants (ChatGPT, Claude, etc.) for syntax, debugging, and code
snippets. You must supply the biological reasoning, the justification for your choices,
and the critical evaluation of what came back. **"The AI said so" is not an answer.**

**Note:** two prompts below (B-factor flexibility, and conserved regions) are
predict-first: you write your prediction down *before* you run anything. That written
prediction is the graded artefact. Being wrong costs you nothing; not having made a
prediction costs you the marks.

## Setup

Same environment as the guide — run this first.

In [ ]:
# Check if running on Google Colab
try:
    from google.colab import drive
    is_google_colab = True
except ImportError:
    is_google_colab = False

# If on Google Colab, install the package
if is_google_colab:
    %pip install numpy==2.1.3 scipy==1.16.3 pandas==2.2.3 biopandas==0.4.1 pypdb==2.4 tqdm==4.67.3 py3dmol==2.4.0

# NOTE: Ignore specific warning message from ipykernel=5.5.6
import warnings
import os

In [ ]:
# Import libraries
import math
import requests
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pypdb
from biopandas.pdb import PandasPdb
import py3Dmol
from tqdm import tqdm


# Suppress all warnings at the Python level
warnings.filterwarnings('ignore')

# Also set environment variable to suppress warnings
os.environ['PYTHONWARNINGS'] = 'ignore'

print("All libraries loaded successfully")

---

## Character Sheet — your capstone protein

Your protein is **assigned**, not chosen freely, and the assignment comes with **its
ligand already identified**. That pairing is the spine of the course: the same protein and
ligand run through ex02 (AlphaFold), ex03 (molecular dynamics), ex04 (docking), and your
final project.

Record your assignment:

- **PDB ID:**
- **Protein name:**
- **Organism:**
- **Method and resolution** (X-ray? NMR? cryo-EM? at what resolution?):
- **Assigned ligand (CCD code and name):**


### Now catalogue *everything* in the structure

Your assignment names **one** ligand. Your structure almost certainly contains **more than
one** non-protein species, and telling them apart is the point of this section.

List **every** heteroatom species in your entry — the way the guide does for 1FSZ — with
how many atoms and how many copies of each. Then classify each one:

| Code | Name | Atoms | Copies | Biological, or experimental artifact? | Evidence |
|---|---|---|---|---|---|
| | | | | | |

**"Evidence" is the graded column.** For each species, say *how you decided*, using more
than one line of reasoning:

- What does the **RCSB entry page or the paper** say the structure was solved to study?
- **Where does it sit** — buried in a pocket making several contacts, or perched in a
  surface groove?
- Do you **recognise it as a lab reagent**? (glycerol, ethylene glycol, HEPES, MES, PEG,
  sulfate, acetate, DMSO…)
- Does it **recur in related structures** of the same protein, or appear only here?
- What do its **occupancy and B-factors** suggest about how well-ordered it is?

**Then find the binding site of your assigned ligand:** compute which residues lie within
4 Å of it, as the guide does for GDP. Keep that list — ex03 and ex04 both build on it.

> **My assigned ligand's contact residues:**

**If your structure is NMR**, there may be no heteroatoms at all, and there will be no
B-factors or occupancies — several rows of the table above will not apply. Say so
explicitly and explain what you can and cannot determine from an NMR ensemble. That is a
real, informative answer, not a failure to complete the task.

**If a species surprises you** — something you cannot classify either way — say so. "I
could not determine whether X is biological, and here is what I checked" is worth more
than a confident guess.

---

## 1. Programmatic query, rooted in your own protein

Fetch your protein's own sequence, then run a sequence search rooted on it to find
related structures — the same pattern as the "fetch 1FSZ's own sequence" and
"build a sequence search" cells in the guide.

**Report:** how many related structures did you find? Is your own structure the top hit?
If it isn't, that is itself worth a sentence — what does it mean?

---

## 2. Structural-quality classification

Batch-process your hits (`extract_structure_info`) and apply the same `classify_quality`
resolution tiers — or your own variant, if you justify why the tiers should differ for
your protein — to build a real table for your own hit set.

---

## 3. Critical evaluation: drug design vs. dynamics

Using **your own** classification table: which structure would you pick for a drug-design
question, and which for a flexibility question?

Justify both choices explicitly, citing actual resolution / R-free / method values from
your table. The guide's version is a model of the *reasoning* — don't restate its
conclusions, they were about a different protein.

---

## 4. Visualization

Produce, with a sentence on what each one shows you:

- Cartoon view coloured by secondary structure
- Cartoon view coloured by domain or motif — **cite where your domain boundaries came
  from** (UniProt, PDB annotation, or a paper)
- B-factor colouring (X-ray) or bundle view (NMR)
- If your structure has a ligand or cofactor: zoom in on it and the residues contacting
  it, as the guide's GDP view does

You will come back to this section: once you have the alignment from section 6, you add a
**conserved-residue view** to this set.

---

## 5. B-factor analysis — predict first

**Before you run anything:** write your prediction here. Which region of your protein do
you expect to be most flexible, and *why*? Base it on the biology — domain architecture,
termini, loops, what the protein has to do.

> **My prediction:**
>
> **My reasoning:**

Then run the B-factor analysis and compare. Were you right? What does the actual pattern
tell you biologically? An honest "I was wrong, and here's what I think I missed" is worth
full marks.

---

## 6. Multiple sequence alignment, and conservation on the structure (required)

This is the section that ties the whole notebook together: you build an alignment, then
**put its result back onto your 3D structure**.

**Step 1 — build the alignment.** Sequence-search hit list → select up to 10 sequences
spread across the ranked list → submit to EBI Clustal Omega → retrieve the aligned FASTA.

**Step 2 — predict first, then compute.** Where would you *expect* conservation to
cluster — an active site, a ligand pocket, an interface? Write that down before you
compute anything.

> **My prediction:**

Then compute per-column conservation anchored on **your** structure's residue numbering,
and report how many positions pass your threshold.

**Step 3 — plot the conserved regions on your figures.** This is the required output, not
an optional extra: take the conserved positions you just identified and **render them on
the structure**, the way the guide highlights 1FSZ's conserved residues alongside its
bound GDP. Add that view to your section 4 figures. If your protein has a ligand,
cofactor or active site, show it in the same view so the reader can see whether
conservation clusters there.

**Step 4 — interpret, against a baseline.** Do the conserved residues actually cluster
where you predicted, or are they spread out?

**Report three numbers, not one:** how many of your ligand's contact residues (from the
Character Sheet section) are conserved, how many you would expect **by chance** given what
fraction of the whole protein is conserved, and the ratio between them. On 1FSZ about 36%
of residues came out conserved, so ~6 of its 17 contacts would be conserved at random — an
overlap has to beat that baseline before it means anything. A result that merely matches
chance is a real finding too: report it as such rather than presenting it as a hit. Does conservation coincide with
the low-B-factor (rigid) regions you found in section 5, or not — and what would either
answer mean?

**Warning: not every protein has FtsZ's 252 relatives.** If your search returns very few,
say so explicitly and explain what *that* means (rarely crystallized? narrow taxonomic
distribution?) rather than forcing the analysis to look like the guide's. Fewer than 3
hits: skip the alignment with a clear note. That is a real, informative outcome — not a
failure.

**Timing:** the EBI queue is shared and slow at busy times; a 10-sequence job can take
several minutes. Start this section early rather than last.

---

## Before you submit

- [ ] Character Sheet filled in (PDB ID, name, organism, why, open question)
- [ ] Both **predictions written before** their analyses (the B-factor prediction in
      section 5, the conservation prediction in section 6)
- [ ] **Conserved residues rendered on your structure**, added to your section 4 figures
- [ ] Every section has a **written interpretation**, not just output
- [ ] Domain boundaries and any external claims are **cited**
- [ ] Anything that didn't work is **described honestly** rather than deleted — a
      documented dead end earns marks, a silently missing section does not